<a href="https://colab.research.google.com/github/predicting-pregnancy-success/pregnancy-ml-models/blob/main/%EB%82%9C%EC%9E%84_%ED%99%98%EC%9E%90_%EB%8C%80%EC%83%81_%EC%9E%84%EC%8B%A0_%EC%84%B1%EA%B3%B5_%EC%97%AC%EB%B6%80_%EC%98%88%EC%B8%A1AI_v4-5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

oong02_ft_cat_tabnet_path = kagglehub.dataset_download('oong02/ft-cat-tabnet')

print('Data source import complete.')


In [ ]:
# ============================================
# ResNet-style MLP (num_workers 수정판)
# ============================================
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

DATA_DIR = '/kaggle/input/datasets/oong02/ft-cat-tabnet'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

train = pd.read_parquet(f'{DATA_DIR}/train_processed.parquet')
test = pd.read_parquet(f'{DATA_DIR}/test_processed.parquet')

TARGET = '임신 성공 여부'
y = train[TARGET].values
drop_cols = [TARGET, 'ID']
feat_cols = [c for c in train.columns if c not in drop_cols and c in test.columns]
X = train[feat_cols].values.astype(np.float32)
X_test = test[feat_cols].values.astype(np.float32)

print(f"Features: {len(feat_cols)} | Train: {X.shape}, Test: {X_test.shape}")

# ============ Model ============
class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.norm1 = nn.BatchNorm1d(dim)
        self.fc1 = nn.Linear(dim, dim)
        self.norm2 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); h = torch.relu(h); h = self.fc1(h)
        h = self.norm2(h); h = torch.relu(h); h = self.dropout(h); h = self.fc2(h)
        return x + h

class ResNetMLP(nn.Module):
    def __init__(self, in_dim, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Linear(in_dim, hidden)
        self.blocks = nn.ModuleList([ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Sequential(
            nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        x = self.proj(x)
        for blk in self.blocks: x = blk(x)
        return self.head(x).squeeze(-1)

# ============ Dataset ============
class TabDS(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y).float() if y is not None else None
    def __len__(self): return len(self.X)
    def __getitem__(self, i):
        if self.y is not None: return self.X[i], self.y[i]
        return self.X[i]

# ============ 학습 함수 ============
def train_one_fold(X_tr, y_tr, X_val, y_val, X_te, seed=42, epochs=30, bs=512, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_te_s = scaler.transform(X_te).astype(np.float32)

    # ★ num_workers=0 (Kaggle 안정성) ★
    train_dl = DataLoader(TabDS(X_tr_s, y_tr), batch_size=bs, shuffle=True, num_workers=0, pin_memory=True)
    val_dl = DataLoader(TabDS(X_val_s, y_val), batch_size=bs*2, num_workers=0, pin_memory=True)
    test_dl = DataLoader(TabDS(X_te_s), batch_size=bs*2, num_workers=0, pin_memory=True)

    model = ResNetMLP(X_tr.shape[1], hidden=256, n_blocks=3, dropout=0.3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0
    best_val = None
    best_test = None
    patience = 5
    no_improve = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _ in val_dl:
                val_preds.append(torch.sigmoid(model(xb.to(device, non_blocking=True))).cpu().numpy())
        val_pred = np.concatenate(val_preds)
        val_auc = roc_auc_score(y_val, val_pred)

        if val_auc > best_auc:
            best_auc = val_auc
            best_val = val_pred
            test_preds = []
            with torch.no_grad():
                for xb in test_dl:
                    test_preds.append(torch.sigmoid(model(xb.to(device, non_blocking=True))).cpu().numpy())
            best_test = np.concatenate(test_preds)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stop at epoch {ep+1}")
                break

        if (ep+1) % 5 == 0:
            print(f"  Epoch {ep+1}: val AUC = {val_auc:.4f} (best {best_auc:.4f})")

    return best_val, best_test, best_auc

# ============ 5-fold CV ============
N_SPLITS = 5
SEED = 42

oof_mlp = np.zeros(len(X))
test_mlp = np.zeros(len(X_test))

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/{N_SPLITS} ---")
    val_pred, test_pred, fold_auc = train_one_fold(
        X[tr_idx], y[tr_idx], X[val_idx], y[val_idx], X_test, seed=SEED+fold)
    oof_mlp[val_idx] = val_pred
    test_mlp += test_pred / N_SPLITS
    print(f"  Fold {fold+1} 최종 AUC: {fold_auc:.4f}")

# ============ 결과 ============
auc_mlp = roc_auc_score(y, oof_mlp)
print(f"\n{'='*60}")
print(f" ResNet-MLP 최종 OOF AUC: {auc_mlp:.4f}")
print('='*60)

oof_cat = np.load(f'{DATA_DIR}/tuned_oof.npy')
oof_ft = np.load(f'{DATA_DIR}/oof_ft.npy')
print(f"\n모델 간 상관계수:")
print(f"  Cat vs MLP : {np.corrcoef(oof_cat, oof_mlp)[0,1]:.4f}  ← 낮을수록 좋음")
print(f"  FT  vs MLP : {np.corrcoef(oof_ft, oof_mlp)[0,1]:.4f}")
print(f"  Cat vs FT  : {np.corrcoef(oof_cat, oof_ft)[0,1]:.4f}  ← 비교 기준")

print(f"\n앙상블 가중치 탐색 (Cat + MLP):")
for w in np.arange(0.4, 0.91, 0.05):
    auc = roc_auc_score(y, w * oof_cat + (1-w) * oof_mlp)
    print(f"  Cat*{w:.2f} + MLP*{1-w:.2f}: {auc:.4f}")

np.save('/kaggle/working/oof_mlp.npy', oof_mlp)
np.save('/kaggle/working/test_mlp.npy', test_mlp)
print("\n 저장 완료")

Device: cuda
Features: 102 | Train: (256351, 102), Test: (90067, 102)

--- Fold 1/5 ---
  Epoch 5: val AUC = 0.7353 (best 0.7354)
  Epoch 10: val AUC = 0.7352 (best 0.7362)
  Early stop at epoch 13
  Fold 1 최종 AUC: 0.7362

--- Fold 2/5 ---
  Epoch 5: val AUC = 0.7407 (best 0.7407)
  Epoch 10: val AUC = 0.7402 (best 0.7412)
  Early stop at epoch 13
  Fold 2 최종 AUC: 0.7412

--- Fold 3/5 ---
  Epoch 5: val AUC = 0.7386 (best 0.7386)
  Epoch 10: val AUC = 0.7386 (best 0.7388)
  Early stop at epoch 12
  Fold 3 최종 AUC: 0.7388

--- Fold 4/5 ---
  Epoch 5: val AUC = 0.7358 (best 0.7358)
  Early stop at epoch 10
  Fold 4 최종 AUC: 0.7358

--- Fold 5/5 ---
  Epoch 5: val AUC = 0.7390 (best 0.7390)
  Epoch 10: val AUC = 0.7383 (best 0.7393)
  Epoch 15: val AUC = 0.7388 (best 0.7394)
  Early stop at epoch 16
  Fold 5 최종 AUC: 0.7394

🎯 ResNet-MLP 최종 OOF AUC: 0.7381

모델 간 상관계수:
  Cat vs MLP : 0.9833  ← 낮을수록 좋음
  FT  vs MLP : 0.9753
  Cat vs FT  : 0.9871  ← 비교 기준

앙상블 가중치 탐색 (Cat + MLP):
  Cat*0.40 + M

In [ ]:
# ============================================
# 3-way 앙상블: Cat + FT + MLP
# ============================================
import numpy as np
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

DATA_DIR = '/kaggle/input/datasets/oong02/ft-cat-tabnet'

# 모든 OOF/Test 로드
oof_cat = np.load(f'{DATA_DIR}/tuned_oof.npy')
oof_ft = np.load(f'{DATA_DIR}/oof_ft.npy')
oof_mlp = np.load('/kaggle/working/oof_mlp.npy')

test_cat = np.load(f'{DATA_DIR}/tuned_test.npy')
test_ft = np.load(f'{DATA_DIR}/test_ft.npy')
test_mlp = np.load('/kaggle/working/test_mlp.npy')

# y 로드
import pandas as pd
train = pd.read_parquet(f'{DATA_DIR}/train_processed.parquet')
y = train['임신 성공 여부'].values

# 기준선
auc_baseline = roc_auc_score(y, 0.65 * oof_cat + 0.35 * oof_ft)
print(f"Baseline (Cat+FT): {auc_baseline:.4f}")
print(f"Cat 단독          : {roc_auc_score(y, oof_cat):.4f}")
print(f"FT  단독          : {roc_auc_score(y, oof_ft):.4f}")
print(f"MLP 단독          : {roc_auc_score(y, oof_mlp):.4f}")

# 1. Coarse grid (3D 가중치 탐색)
print("\n" + "="*60)
print("Coarse Grid Search (3-way)")
print("="*60)
best_auc = 0
best_w = None
for w_cat in np.arange(0.3, 0.81, 0.1):
    for w_ft in np.arange(0.1, 0.61, 0.1):
        w_mlp = 1 - w_cat - w_ft
        if w_mlp < 0 or w_mlp > 0.6:
            continue
        oof_blend = w_cat * oof_cat + w_ft * oof_ft + w_mlp * oof_mlp
        auc = roc_auc_score(y, oof_blend)
        if auc > best_auc:
            best_auc = auc
            best_w = (w_cat, w_ft, w_mlp)
        if auc > auc_baseline:
            print(f"  Cat={w_cat:.2f} FT={w_ft:.2f} MLP={w_mlp:.2f}: {auc:.4f}  ⭐")

print(f"\n최고: Cat={best_w[0]:.2f} FT={best_w[1]:.2f} MLP={best_w[2]:.2f} → {best_auc:.4f}")

# 2. Fine grid 주변 탐색
print("\n" + "="*60)
print("Fine Grid Search (best 주변)")
print("="*60)
w_cat0, w_ft0, w_mlp0 = best_w
for w_cat in np.arange(max(0.01, w_cat0-0.08), w_cat0+0.09, 0.02):
    for w_ft in np.arange(max(0.01, w_ft0-0.08), w_ft0+0.09, 0.02):
        w_mlp = 1 - w_cat - w_ft
        if w_mlp < 0:
            continue
        oof_blend = w_cat * oof_cat + w_ft * oof_ft + w_mlp * oof_mlp
        auc = roc_auc_score(y, oof_blend)
        if auc > best_auc:
            best_auc = auc
            best_w = (w_cat, w_ft, w_mlp)

print(f"\n최종 최적: Cat={best_w[0]:.3f} FT={best_w[1]:.3f} MLP={best_w[2]:.3f}")
print(f"최종 OOF: {best_auc:.4f}")
print(f"Baseline 대비: {best_auc - auc_baseline:+.4f}")

# 3. Scipy로 연속 최적화 (negative AUC 최소화)
print("\n" + "="*60)
print("Scipy Optimization (연속 최적화)")
print("="*60)

def neg_auc(weights, oofs, y):
    weights = np.abs(weights) / np.sum(np.abs(weights))  # softmax-like normalization
    blend = sum(w * oof for w, oof in zip(weights, oofs))
    return -roc_auc_score(y, blend)

from scipy.optimize import minimize
best_scipy_auc = 0
best_scipy_w = None
for trial in range(10):  # 여러 초기값 시도
    np.random.seed(trial)
    x0 = np.random.dirichlet([1, 1, 1])
    result = minimize(
        neg_auc, x0,
        args=([oof_cat, oof_ft, oof_mlp], y),
        method='Nelder-Mead',
        options={'maxiter': 200, 'xatol': 1e-5}
    )
    w = np.abs(result.x) / np.sum(np.abs(result.x))
    auc = -result.fun
    if auc > best_scipy_auc:
        best_scipy_auc = auc
        best_scipy_w = w

print(f"Scipy 최적: Cat={best_scipy_w[0]:.3f} FT={best_scipy_w[1]:.3f} MLP={best_scipy_w[2]:.3f}")
print(f"Scipy OOF: {best_scipy_auc:.4f}")

# 최종 선택
final_auc = max(best_auc, best_scipy_auc)
final_w = best_w if best_auc >= best_scipy_auc else best_scipy_w

print("\n" + "="*60)
print(f"🎯 최종 결정")
print("="*60)
print(f"가중치: Cat={final_w[0]:.3f}, FT={final_w[1]:.3f}, MLP={final_w[2]:.3f}")
print(f"OOF: {final_auc:.4f}")
print(f"Baseline 0.7408 대비: {final_auc - 0.7408:+.4f}")

# Submission 생성
test_final = final_w[0] * test_cat + final_w[1] * test_ft + final_w[2] * test_mlp
sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
sub['probability'] = test_final
sub.to_csv('/kaggle/working/submission_3way.csv', index=False)
print(f"\n예측 분포:")
print(sub['probability'].describe())
print(f"\n✅ submission_3way.csv 저장 완료")

Baseline (Cat+FT): 0.7408
Cat 단독          : 0.7404
FT  단독          : 0.7395
MLP 단독          : 0.7381

Coarse Grid Search (3-way)
  Cat=0.30 FT=0.40 MLP=0.30: 0.7408  ⭐
  Cat=0.40 FT=0.30 MLP=0.30: 0.7409  ⭐
  Cat=0.40 FT=0.40 MLP=0.20: 0.7409  ⭐
  Cat=0.50 FT=0.20 MLP=0.30: 0.7408  ⭐
  Cat=0.50 FT=0.30 MLP=0.20: 0.7409  ⭐
  Cat=0.50 FT=0.40 MLP=0.10: 0.7409  ⭐
  Cat=0.60 FT=0.20 MLP=0.20: 0.7409  ⭐
  Cat=0.60 FT=0.30 MLP=0.10: 0.7409  ⭐
  Cat=0.70 FT=0.20 MLP=0.10: 0.7409  ⭐

최고: Cat=0.50 FT=0.30 MLP=0.20 → 0.7409

Fine Grid Search (best 주변)

최종 최적: Cat=0.520 FT=0.300 MLP=0.180
최종 OOF: 0.7409
Baseline 대비: +0.0001

Scipy Optimization (연속 최적화)
Scipy 최적: Cat=0.520 FT=0.306 MLP=0.174
Scipy OOF: 0.7409

🎯 최종 결정
가중치: Cat=0.520, FT=0.306, MLP=0.174
OOF: 0.7409
Baseline 0.7408 대비: +0.0001

예측 분포:
count    90067.000000
mean         0.259648
std          0.160692
min          0.000312
25%          0.142919
50%          0.272723
75%          0.380642
max          0.738237
Name: probability, dtype

In [ ]:
# 함수만 다시 정의 (이전 셀들 그대로 유지)
def train_one_fold_v2(X_tr, y_tr, X_val, y_val, X_te, seed=42, epochs=60, bs=512, lr=8e-4):
    torch.manual_seed(seed); np.random.seed(seed)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr).astype(np.float32)
    X_val_s = scaler.transform(X_val).astype(np.float32)
    X_te_s = scaler.transform(X_te).astype(np.float32)

    train_dl = DataLoader(TabDS(X_tr_s, y_tr), batch_size=bs, shuffle=True, num_workers=0, pin_memory=True)
    val_dl = DataLoader(TabDS(X_val_s, y_val), batch_size=bs*2, num_workers=0, pin_memory=True)
    test_dl = DataLoader(TabDS(X_te_s), batch_size=bs*2, num_workers=0, pin_memory=True)

    model = ResNetMLP(X_tr.shape[1], hidden=256, n_blocks=3, dropout=0.3).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0; best_val = None; best_test = None
    patience = 15; no_improve = 0

    for ep in range(epochs):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward(); optimizer.step()
        scheduler.step()

        model.eval()
        val_preds = []
        with torch.no_grad():
            for xb, _ in val_dl:
                val_preds.append(torch.sigmoid(model(xb.to(device, non_blocking=True))).cpu().numpy())
        val_pred = np.concatenate(val_preds)
        val_auc = roc_auc_score(y_val, val_pred)

        if val_auc > best_auc:
            best_auc = val_auc; best_val = val_pred
            test_preds = []
            with torch.no_grad():
                for xb in test_dl:
                    test_preds.append(torch.sigmoid(model(xb.to(device, non_blocking=True))).cpu().numpy())
            best_test = np.concatenate(test_preds)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"  Early stop at epoch {ep+1}")
                break

        if (ep+1) % 10 == 0:
            print(f"  Epoch {ep+1}: val AUC = {val_auc:.4f} (best {best_auc:.4f})")

    return best_val, best_test, best_auc

# v2로 재학습
oof_mlp_v2 = np.zeros(len(X))
test_mlp_v2 = np.zeros(len(X_test))
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Fold {fold+1}/5 (v2) ---")
    val_pred, test_pred, fold_auc = train_one_fold_v2(
        X[tr_idx], y[tr_idx], X[val_idx], y[val_idx], X_test, seed=42+fold)
    oof_mlp_v2[val_idx] = val_pred
    test_mlp_v2 += test_pred / 5
    print(f"  Fold {fold+1} AUC: {fold_auc:.4f}")

auc_mlp_v2 = roc_auc_score(y, oof_mlp_v2)
print(f"\n MLP v2 OOF: {auc_mlp_v2:.4f} (v1: 0.7381)")
print(f"  Cat 상관 v2: {np.corrcoef(oof_cat, oof_mlp_v2)[0,1]:.4f} (v1: 0.9833)")

# v2로 3-way
print("\n=== v2 3-way 탐색 ===")
best_auc_v2 = 0; best_w_v2 = None
for w_cat in np.arange(0.3, 0.81, 0.05):
    for w_ft in np.arange(0.05, 0.61, 0.05):
        w_mlp = 1 - w_cat - w_ft
        if w_mlp < 0 or w_mlp > 0.6: continue
        auc = roc_auc_score(y, w_cat * oof_cat + w_ft * oof_ft + w_mlp * oof_mlp_v2)
        if auc > best_auc_v2:
            best_auc_v2 = auc; best_w_v2 = (w_cat, w_ft, w_mlp)

print(f"v2 3-way 최고: {best_auc_v2:.4f} (v1: 0.7409)")
print(f"가중치: Cat={best_w_v2[0]:.2f}, FT={best_w_v2[1]:.2f}, MLP={best_w_v2[2]:.2f}")

# v2가 더 좋으면 submission 업데이트
if best_auc_v2 > 0.7409:
    test_final_v2 = best_w_v2[0] * test_cat + best_w_v2[1] * test_ft + best_w_v2[2] * test_mlp_v2
    sub = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
    sub['probability'] = test_final_v2
    sub.to_csv('/kaggle/working/submission_3way_v2.csv', index=False)
    print(f"\n submission_3way_v2.csv 생성 (더 좋은 버전)")
else:
    print(f"\n v2가 v1보다 안 좋음. v1 유지.")


--- Fold 1/5 (v2) ---
  Epoch 10: val AUC = 0.7358 (best 0.7360)
  Epoch 20: val AUC = 0.7336 (best 0.7360)
  Early stop at epoch 22
  Fold 1 AUC: 0.7360

--- Fold 2/5 (v2) ---
  Epoch 10: val AUC = 0.7405 (best 0.7407)
  Epoch 20: val AUC = 0.7388 (best 0.7411)
  Early stop at epoch 26
  Fold 2 AUC: 0.7411

--- Fold 3/5 (v2) ---
  Epoch 10: val AUC = 0.7387 (best 0.7387)
  Epoch 20: val AUC = 0.7371 (best 0.7393)
  Early stop at epoch 26
  Fold 3 AUC: 0.7393

--- Fold 4/5 (v2) ---
  Epoch 10: val AUC = 0.7354 (best 0.7359)
  Early stop at epoch 20
  Fold 4 AUC: 0.7359

--- Fold 5/5 (v2) ---
  Epoch 10: val AUC = 0.7385 (best 0.7395)
  Epoch 20: val AUC = 0.7367 (best 0.7395)
  Early stop at epoch 23
  Fold 5 AUC: 0.7395

 MLP v2 OOF: 0.7381 (v1: 0.7381)
  Cat 상관 v2: 0.9830 (v1: 0.9833)

=== v2 3-way 탐색 ===
v2 3-way 최고: 0.7409 (v1: 0.7409)
가중치: Cat=0.50, FT=0.30, MLP=0.20

 submission_3way_v2.csv 생성 (더 좋은 버전)
